# XGBoost Simulation — Shredder Blade Wear Rate (RUL) Prediction

## 개요

이 노트북은 **XGBoost(Gradient Boosting)**를 이용한 칼날 마모율 예측을 체험하는 시뮬레이션입니다.

| 항목 | 내용 |
|------|------|
| **모델** | GradientBoostingRegressor (scikit-learn) |
| **시나리오** | 슈레더 칼날 마모율(0~100%) 예측 — 500시간 교체 주기 |
| **핵심** | 특징 엔지니어링(Feature Engineering) + 해석 가능성(Feature Importance) |

### XGBoost의 핵심 원리

```
원본 시계열 → 특징 엔지니어링 → 30개+ 특징 생성 → Gradient Boosting 학습
                                                    ↓
                               칼날 마모율 예측 + 특징 중요도 해석
```

### 다른 모델과의 비교

| 비교 | Prophet | LSTM | **XGBoost** |
|:---:|:---:|:---:|:---:|
| 입력 변수 | 1개 (단변량) | 3개 (다변량) | **30개+ (특징 엔지니어링)** |
| 목적 | 미래 값 예측 | 이상 탐지 | **마모율(RUL) 예측** |
| 해석성 | 높음 (분해) | 낮음 (블랙박스) | **높음 (특징 중요도)** |
| 속도 | 빠름 | 느림 (GPU 필요) | **가장 빠름 (Edge 배포 최적)** |
| 특징 설계 | 자동 | 자동 | **수동 (도메인 지식 필요)** |

---
## Step 0. 라이브러리 설치 및 Import

In [ ]:
!pip install -q scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('All libraries loaded successfully!')

---
## Step 1. Shredder Sensor Data Generation + Blade Wear Simulation

산업용 슈레더(Shredder)의 센서 데이터를 시뮬레이션하고, 칼날 마모율(예측 대상)을 생성합니다.

### 생성되는 센서 데이터

| 센서 | 단위 | 정상 범위 | 설명 |
|------|------|-----------|------|
| temperature | °C | 20~38 | 베어링 온도 |
| vibration | mm/s | 1.5~4.0 | 진동 RMS |
| current | A | 65~110 | 모터 전류 |
| rpm | rpm | 19~21 | 회전 속도 |
| throughput | t/h | 1.5~3.5 | 처리량 |

### 칼날 마모율 (blade_wear)

- 시간에 따라 선형 증가 (0→100%)
- 전류/진동이 높을수록 마모 가속
- **500시간 주기로 칼날 교체** (마모율 리셋)

In [ ]:
def generate_shredder_data(days=90, freq_minutes=10, seed=42):
    """
    Shredder sensor data simulator.
    Generates realistic bearing temperature, vibration, current, rpm, throughput.
    """
    np.random.seed(seed)
    n_points = days * 24 * 60 // freq_minutes
    timestamps = pd.date_range(start='2026-01-01', periods=n_points, freq=f'{freq_minutes}min')
    t = np.arange(n_points)
    hours = np.array([ts.hour for ts in timestamps])
    dow = np.array([ts.dayofweek for ts in timestamps])

    base_temp = 28.0
    daily_pattern = 5.0 * np.sin(2 * np.pi * hours / 24 - np.pi/2)
    weekly_pattern = np.where(dow >= 5, -3.0, 0.0)
    wear_trend = 0.03 * t / (24 * 60 / freq_minutes)
    noise = np.random.normal(0, 0.8, n_points)
    anomaly_mask = np.random.random(n_points) < 0.005
    anomaly_spike = anomaly_mask * np.random.uniform(15, 30, n_points)
    temperature = np.clip(base_temp + daily_pattern + weekly_pattern + wear_trend + noise + anomaly_spike, 15, 80)

    base_vib = 2.5
    vib_daily = 0.5 * np.sin(2 * np.pi * hours / 24)
    vib_wear = 0.02 * t / (24 * 60 / freq_minutes)
    vib_noise = np.random.normal(0, 0.3, n_points)
    vib_anomaly = anomaly_mask * np.random.uniform(5, 15, n_points)
    vibration = np.clip(base_vib + vib_daily + vib_wear + vib_noise + vib_anomaly, 0.5, 25)

    base_cur = 85.0
    cur_daily = 10.0 * np.sin(2 * np.pi * hours / 24 - np.pi/3)
    cur_wear = 0.05 * t / (24 * 60 / freq_minutes)
    cur_noise = np.random.normal(0, 2.0, n_points)
    cur_weekend = np.where(dow >= 5, -30.0, 0.0)
    current = np.clip(base_cur + cur_daily + cur_wear + cur_noise + cur_weekend, 20, 150)

    rpm = np.clip(20.0 + np.random.normal(0, 0.3, n_points) + np.where(dow >= 5, -15.0, 0.0), 0, 25)
    throughput = np.clip(2.5 + 0.5 * np.sin(2*np.pi*hours/24 - np.pi/4) + np.random.normal(0, 0.15, n_points) + np.where(dow >= 5, -2.0, 0.0), 0, 4)

    df = pd.DataFrame({
        'timestamp': timestamps,
        'temperature': np.round(temperature, 2),
        'vibration': np.round(vibration, 2),
        'current': np.round(current, 2),
        'rpm': np.round(rpm, 2),
        'throughput': np.round(throughput, 2)
    })
    return df

print('generate_shredder_data() defined.')

In [ ]:
def simulate_blade_wear(df):
    """
    Blade wear rate simulation (0~100%).
    Linear wear + current/vibration acceleration + 500h replacement cycle.
    """
    n = len(df)
    # Base wear: linear increase over time
    base_wear = np.linspace(0, 100, n)

    # Higher current/vibration = faster wear
    cur_effect = (df['current'].values - 85) * 0.01
    vib_effect = (df['vibration'].values - 2.5) * 0.05

    wear = base_wear + np.cumsum(cur_effect + vib_effect) * 0.01
    wear = np.clip(wear, 0, 100)

    # 500-hour blade replacement cycle (wear reset)
    cycle_length = 500
    wear_cycles = wear % cycle_length / cycle_length * 100

    return np.round(wear_cycles, 2)

print('simulate_blade_wear() defined.')

In [ ]:
# Generate 90 days of data, sampled every 1 hour
df = generate_shredder_data(days=90, freq_minutes=60)
df['blade_wear'] = simulate_blade_wear(df)

print(f'Generated data: {len(df)} samples')
print(f'Period: {df["timestamp"].min()} ~ {df["timestamp"].max()}')
print(f'Blade wear range: {df["blade_wear"].min():.1f}% ~ {df["blade_wear"].max():.1f}%')
print(f'\nStatistics:')
df.describe().round(2)

---
## Step 2. Raw Data Visualization

생성된 원본 데이터를 확인합니다.

**관찰 포인트:**
- 5개 센서의 시간에 따른 변화 패턴
- 칼날 마모율의 500시간 주기적 리셋 패턴
- 센서 값과 마모율의 상관관계

In [ ]:
# 2-1. Sensor data overview (5 sensors)
fig, axes = plt.subplots(5, 1, figsize=(14, 16), sharex=True)

sensors = [
    ('temperature', 'Bearing Temperature', 'C', 'tab:red'),
    ('vibration', 'Vibration RMS', 'mm/s', 'tab:blue'),
    ('current', 'Motor Current', 'A', 'tab:green'),
    ('rpm', 'RPM', 'rpm', 'tab:orange'),
    ('throughput', 'Throughput', 't/h', 'tab:purple'),
]

for ax, (col, title, unit, color) in zip(axes, sensors):
    ax.plot(df['timestamp'], df[col], color=color, alpha=0.7, linewidth=0.5)
    ax.set_ylabel(f'{title} ({unit})', fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_title(f'{title}', fontsize=12, fontweight='bold')

axes[-1].set_xlabel('Date', fontsize=11)
fig.suptitle('Shredder Sensor Data — 90 Days Overview', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 2-2. Blade wear over time
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['timestamp'], df['blade_wear'], 'b-', linewidth=0.8, alpha=0.8)
ax.axhline(y=80, color='red', linestyle='--', linewidth=2, label='Replacement threshold (80%)')
ax.set_title('Blade Wear Rate Over Time (500h Replacement Cycles)', fontsize=14, fontweight='bold')
ax.set_ylabel('Blade Wear (%)')
ax.set_xlabel('Date')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'500\uc2dc\uac04 \uad50\uccb4 \uc8fc\uae30\uac00 \ubcf4\uc774\ub294 \ud1b1\ub2c8 \ubaa8\uc591\uc758 \ub9c8\ubaa8\uc728 \ud328\ud134\uc744 \ud655\uc778\ud558\uc138\uc694.')

In [ ]:
# 2-3. Wear vs sensor correlations
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

pairs = [
    ('temperature', 'Bearing Temperature (C)'),
    ('vibration', 'Vibration RMS (mm/s)'),
    ('current', 'Motor Current (A)'),
]

for ax, (col, label) in zip(axes, pairs):
    ax.scatter(df[col], df['blade_wear'], alpha=0.1, s=3, color='steelblue')
    ax.set_xlabel(label)
    ax.set_ylabel('Blade Wear (%)')
    ax.set_title(f'Wear vs {col.capitalize()}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)

fig.suptitle('Blade Wear vs Sensor Correlations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 3. Feature Engineering (XGBoost Core!)

XGBoost 시계열의 **핵심**은 특징 엔지니어링입니다.

LSTM은 원본 시계열을 그대로 넣으면 자동으로 패턴을 학습하지만,
XGBoost는 **수동으로 의미 있는 특징을 설계**해야 합니다.

### 생성하는 특징 유형

| 유형 | 예시 | 설명 |
|------|------|------|
| 시간 특징 | hour, day_of_week | 일간/주간 패턴 포착 |
| 센서 현재값 | temperature, vibration, current | 현재 상태 |
| Lag 특징 | temp_lag1h, temp_lag24h | 과거 값 참조 |
| Rolling 통계 | rolling_mean_6h, rolling_std_6h | 최근 통계 |
| 변화율 | temp_diff_1h | 변화 속도 |
| 운전 정보 | rpm, throughput, operating_hours | 운전 조건 |

In [ ]:
def create_features(df):
    """
    Convert time series to XGBoost input features.
    This is the CORE of XGBoost time series — manual feature engineering.
    """
    feat = pd.DataFrame()

    # Time features
    feat['hour'] = df['timestamp'].dt.hour
    feat['day_of_week'] = df['timestamp'].dt.dayofweek
    feat['day_of_month'] = df['timestamp'].dt.day

    for col in ['temperature', 'vibration', 'current']:
        # Current sensor values
        feat[f'{col}'] = df[col].values

        # Lag features — "value from 1h, 6h, 24h ago"
        for lag in [1, 6, 24]:
            feat[f'{col}_lag{lag}h'] = df[col].shift(lag).values

        # Rolling statistics — "mean/std over last 6h, mean over last 24h"
        feat[f'{col}_rolling_mean_6h'] = df[col].rolling(6).mean().values
        feat[f'{col}_rolling_std_6h'] = df[col].rolling(6).std().values
        feat[f'{col}_rolling_mean_24h'] = df[col].rolling(24).mean().values

        # Diff features — "1h change rate"
        feat[f'{col}_diff_1h'] = df[col].diff(1).values

    # RPM, Throughput
    feat['rpm'] = df['rpm'].values
    feat['throughput'] = df['throughput'].values

    # Operating hours (cumulative)
    feat['operating_hours'] = np.arange(len(df))

    return feat

print('create_features() defined.')

In [ ]:
# Create features
features = create_features(df)
target = df['blade_wear']

# Remove NaN rows (caused by lag/rolling)
valid_mask = features.notna().all(axis=1)
features = features[valid_mask]
target = target[valid_mask]

feature_names = features.columns.tolist()
print(f'\uc0dd\uc131\ub41c \ud2b9\uc9d5: {len(feature_names)}\uac1c')
print('=' * 40)
for i, name in enumerate(feature_names, 1):
    print(f'  {i:2d}. {name}')

print(f'\n\u2605 Prophet: \uc2dc\uac04+\uc628\ub3c4 2\uac1c\ub9cc \uc0ac\uc6a9')
print(f'\u2605 XGBoost: {len(feature_names)}\uac1c \ud2b9\uc9d5 \uc0ac\uc6a9 \u2192 \ub354 \uc815\ud655!')

---
## Step 4. Train / Val / Test Split

시계열 데이터는 **반드시 시간순으로 분할**해야 합니다.

```
|<--- Train (70%) --->|<- Val (15%) ->|<- Test (15%) ->|
|   Jan ~ early Mar   |  mid Mar      |  late Mar      |
```

3분할 이유: Validation으로 하이퍼파라미터 튜닝, Test로 최종 성능 평가

In [ ]:
# Time-ordered 70/15/15 split
train_size = int(len(features) * 0.7)
val_size = int(len(features) * 0.15)

X_train = features.iloc[:train_size]
y_train = target.iloc[:train_size]
X_val = features.iloc[train_size:train_size+val_size]
y_val = target.iloc[train_size:train_size+val_size]
X_test = features.iloc[train_size+val_size:]
y_test = target.iloc[train_size+val_size:]

print(f'Train: {len(X_train)} samples')
print(f'Val  : {len(X_val)} samples')
print(f'Test : {len(X_test)} samples')
print(f'Total: {len(X_train) + len(X_val) + len(X_test)} samples')

In [ ]:
# Visualize train/val/test split
valid_timestamps = df['timestamp'][valid_mask].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(valid_timestamps.iloc[:train_size], y_train.values, 'b.', alpha=0.3, markersize=2, label='Train (70%)')
ax.plot(valid_timestamps.iloc[train_size:train_size+val_size], y_val.values, 'orange', alpha=0.5, marker='.', markersize=2, linestyle='none', label='Val (15%)')
ax.plot(valid_timestamps.iloc[train_size+val_size:], y_test.values, 'g.', alpha=0.5, markersize=2, label='Test (15%)')
ax.axvline(x=valid_timestamps.iloc[train_size], color='red', linestyle='--', linewidth=2, label='Train/Val boundary')
ax.axvline(x=valid_timestamps.iloc[train_size+val_size], color='darkgreen', linestyle='--', linewidth=2, label='Val/Test boundary')
ax.set_title('Train / Val / Test Split (70% / 15% / 15%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Blade Wear (%)')
ax.set_xlabel('Date')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 5. Model Training

**GradientBoostingRegressor** 모델을 학습합니다.

| 파라미터 | 값 | 설명 |
|------------|------|------|
| n_estimators | 200 | 부스팅 트리 수 |
| max_depth | 6 | 트리 최대 깊이 |
| learning_rate | 0.1 | 학습률 |
| subsample | 0.8 | 각 트리에 사용할 데이터 비율 |
| random_state | 42 | 재현성 |

In [ ]:
model = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42
)

model.fit(X_train, y_train)
print('Model training complete!')
print(f'  n_estimators : {model.n_estimators}')
print(f'  max_depth    : {model.max_depth}')
print(f'  learning_rate: {model.learning_rate}')
print(f'  n_features   : {model.n_features_in_}')

---
## Step 6. Performance Evaluation

3가지 지표로 성능을 평가합니다.

| 지표 | 의미 |
|------|------|
| **MAE** | 평균 절대 오차 — 평균적으로 몇 % 틀리는가 |
| **RMSE** | 평균 제곱근 오차 — 큰 오차에 더 민감 |
| **R²** | 결정계수 — 1에 가까울수록 좋음 |

In [ ]:
# Predictions
y_pred_train = model.predict(X_train)
y_pred_val = model.predict(X_val)
y_pred_test = model.predict(X_test)

# Test metrics
mae = mean_absolute_error(y_test, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2 = r2_score(y_test, y_pred_test)

# Val metrics for comparison
val_mae = mean_absolute_error(y_val, y_pred_val)
val_r2 = r2_score(y_val, y_pred_val)

print('=' * 50)
print('  Performance Evaluation')
print('=' * 50)
print(f'  [Validation]')
print(f'    MAE  = {val_mae:.2f}%')
print(f'    R2   = {val_r2:.4f}')
print(f'')
print(f'  [Test]')
print(f'    MAE  = {mae:.2f}%')
print(f'    RMSE = {rmse:.2f}%')
print(f'    R2   = {r2:.4f}')
print('=' * 50)

---
## Step 7. Result Visualization

### 7-1. Prediction vs Actual

테스트 구간의 실제 칼날 마모율과 XGBoost 예측값을 비교합니다.
80% 라인은 칼날 교체 권고선입니다.

In [ ]:
test_timestamps = valid_timestamps.iloc[train_size+val_size:train_size+val_size+len(y_test)]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_timestamps, y_test.values, 'b-', alpha=0.7, linewidth=1, label='Actual Wear')
ax.plot(test_timestamps, y_pred_test, 'r--', alpha=0.8, linewidth=1, label='XGBoost Prediction')
ax.axhline(y=80, color='orange', linestyle=':', linewidth=2, label='Replacement threshold (80%)')
ax.set_title(f'XGBoost Blade Wear Prediction (MAE={mae:.2f}%, R2={r2:.4f})',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Blade Wear (%)')
ax.set_xlabel('Date')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7-2. Feature Importance Top 15

XGBoost의 가장 큰 장점: **"어떤 센서가 마모 예측에 가장 중요한가?"** 를 해석할 수 있습니다.

In [ ]:
importance = pd.Series(model.feature_importances_, index=feature_names)
importance = importance.sort_values(ascending=False)
top15 = importance.head(15)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#dc2626' if v > 0.05 else '#2563eb' if v > 0.02 else '#94a3b8' for v in top15.values]
ax.barh(range(len(top15)), top15.values, color=colors)
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15.index, fontsize=10)
ax.invert_yaxis()
ax.set_title('Feature Importance Top 15 — "Which sensor matters most for wear prediction?"',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print('\ud2b9\uc9d5 \uc911\uc694\ub3c4 \ud574\uc11d:')
print(f'  1\uc704: {importance.index[0]} ({importance.values[0]:.4f})')
print(f'  2\uc704: {importance.index[1]} ({importance.values[1]:.4f})')
print(f'  3\uc704: {importance.index[2]} ({importance.values[2]:.4f})')

### 7-3. Error Distribution

예측 오차(실제 - 예측)의 분포를 히스토그램으로 확인합니다.

In [ ]:
errors = y_test.values - y_pred_test

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(errors, bins=50, color='steelblue', alpha=0.7, edgecolor='white', density=True)
ax.axvline(x=0, color='red', linewidth=2, linestyle='--', label='Zero error')
ax.axvline(x=np.mean(errors), color='orange', linewidth=2,
           label=f'Mean error: {np.mean(errors):.2f}%')
ax.set_title('Prediction Error Distribution (Actual - Predicted)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Error (%)')
ax.set_ylabel('Density')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Error statistics:')
print(f'  Mean  : {np.mean(errors):+.2f}%')
print(f'  Std   : {np.std(errors):.2f}%')
print(f'  Median: {np.median(errors):+.2f}%')
print(f'  Within +/- 5%: {(np.abs(errors) < 5).mean()*100:.1f}%')

### 7-4. Scatter Plot: Actual vs Predicted

실제값과 예측값의 산점도입니다. 대각선에 가까울수록 예측이 정확합니다.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test.values, y_pred_test, alpha=0.3, s=10, color='steelblue')

# Diagonal line (perfect prediction)
min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect prediction')

ax.set_title(f'Actual vs Predicted (R2={r2:.4f})', fontsize=13, fontweight='bold')
ax.set_xlabel('Actual Blade Wear (%)')
ax.set_ylabel('Predicted Blade Wear (%)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

### 7-5. Feature Importance by Group

특징들을 그룹별로 묶어 기여도를 파이 차트로 확인합니다.

- **Temperature 그룹**: temperature, temp_lag, temp_rolling, temp_diff
- **Vibration 그룹**: vibration, vib_lag, vib_rolling, vib_diff
- **Current 그룹**: current, cur_lag, cur_rolling, cur_diff
- **Time 그룹**: hour, day_of_week, day_of_month
- **Operating 그룹**: rpm, throughput, operating_hours

In [ ]:
# Group features by category
groups = {
    'Temperature': [f for f in feature_names if 'temperature' in f or 'temp' in f],
    'Vibration': [f for f in feature_names if 'vibration' in f or 'vib' in f],
    'Current': [f for f in feature_names if 'current' in f or 'cur' in f],
    'Time': ['hour', 'day_of_week', 'day_of_month'],
    'Operating': ['rpm', 'throughput', 'operating_hours'],
}

group_importance = {}
for group_name, group_features in groups.items():
    existing = [f for f in group_features if f in importance.index]
    group_importance[group_name] = importance[existing].sum()

group_series = pd.Series(group_importance).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 8))
colors = ['#dc2626', '#2563eb', '#16a34a', '#f59e0b', '#8b5cf6']
wedges, texts, autotexts = ax.pie(
    group_series.values,
    labels=group_series.index,
    autopct='%1.1f%%',
    colors=colors[:len(group_series)],
    startangle=90,
    textprops={'fontsize': 12}
)
for autotext in autotexts:
    autotext.set_fontsize(11)
    autotext.set_fontweight('bold')

ax.set_title('Feature Importance by Group', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\uadf8\ub8f9\ubcc4 \uc911\uc694\ub3c4:')
for name, val in group_series.items():
    print(f'  {name:15s}: {val:.4f} ({val/group_series.sum()*100:.1f}%)')

---
## Step 8. Summary

### XGBoost 핵심 특징 요약

| 항목 | 내용 |
|------|------|
| **입력** | 30개+ 수동 설계 특징 (lag, rolling, diff 등) |
| **학습 방식** | Gradient Boosting (200개 트리 앱상블) |
| **예측 대상** | 칼날 마모율 (0~100%, 500h 교체 주기) |
| **핵심 강점** | 특징 중요도 → 해석 가능 |

### 장점
- **특징 중요도**: "어떤 센서가 마모에 가장 영향을 주는가?" 확인 가능
- **매우 빠름**: 학습 수 초, 추론 <5ms (Edge 배포 최적)
- **가벼움**: GPU 불필요, CPU만으로 충분
- **Edge 배포**: 모델 파일 크기 작음, 낮은 하드웨어 사양에서도 동작

### 단점
- **수동 특징 엔지니어링 필요**: lag, rolling, diff 등을 도메인 지식으로 설계해야 함
  - LSTM/TFT는 자동으로 시퀀스 패턴 학습
- 매우 긴 시퀀스 패턴 학습에 한계

### 슈레더 적용 권장
- **칼날 마모율(RUL) 예측** → **최적!** (이 프로젝트에서 실제 사용)
- 이유: 해석 가능 + 빠름 + Edge 배포 용이
- 결과를 현장 엔지니어에게 "전류가 가장 중요" 등으로 설명 가능

---

> **다음 단계**: `05_ARIMA` 폴더에서 전통적 통계 모델인 ARIMA를 체험해 보세요.

In [ ]:
print('=' * 60)
print('  XGBoost Simulation Complete!')
print('=' * 60)
print(f'''
  Model: GradientBoostingRegressor
  Features: {len(feature_names)} engineered features

  Top 3 Important Features:
    1. {importance.index[0]:25s} ({importance.values[0]:.4f})
    2. {importance.index[1]:25s} ({importance.values[1]:.4f})
    3. {importance.index[2]:25s} ({importance.values[2]:.4f})

  Performance:
    MAE  = {mae:.2f}%
    RMSE = {rmse:.2f}%
    R2   = {r2:.4f}

  Advantages:
    - Feature importance: interpretable
    - Very fast: training ~seconds, inference <5ms
    - Lightweight: no GPU needed, Edge deployment ready

  Disadvantages:
    - Manual feature engineering required

  Shredder Application:
    - Blade wear RUL prediction = OPTIMAL!
''')